<a href="https://colab.research.google.com/github/manish7725/deeplearning/blob/main/Lecture%2007%20-%20Partial%20Derivatives%2C%20Gradients%20and%20the%20Chain%20Rule/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 07 — Partial Derivatives, Gradients and the Chain Rule · Laboratory

↩ **Theory:** [`blog.md`](<blog.md>) — read the lecture first. This notebook tests it.

## Step 1 — The Problem

Chapter 6 differentiated functions of **one** variable, so Chapter 1 had to pin the bias at
$b=1$ to study $w$ alone. Take the pin out and the loss becomes a **bowl**:

$$L(w_1, b) \quad\text{— walk east and it rises, walk north and it falls}$$

"How steep is it here?" is not a complete question until you say *in which direction*.

## Step 2 — Prediction

Commit before running. You check these in Step 10.

1. At $w_1 = 0$, $b = 0$, is the loss more sensitive to $w_1$ or to $b$?
2. Of all the directions you could step from a point, which one makes the loss fall fastest?
3. The curvature is now a $2\times2$ matrix, not one number. What sets the learning-rate
   ceiling — the largest curvature, the smallest, or the average?
4. Chapter 1 measured a 2-parameter threshold near $0.12$. Can you predict this one exactly?

In [ ]:
# Step 3 — Intuition: one dial at a time.
import numpy as np
np.random.seed(0)

rooms  = np.array([2., 2., 3., 4.])
area   = np.array([800., 1200., 900., 1600.])
prices = np.array([9., 11., 11.5, 17.])

def L(w1, b, w2=0.005):
    """Loss with the area weight pinned at its true value, so we can see a 2-D bowl."""
    pred = w1 * rooms + w2 * area + b
    return np.mean((pred - prices) ** 2)

print("hold b = 0, nudge w1:")
for w1 in [0.0, 0.5, 1.0]:
    print(f"   L({w1}, 0) = {L(w1, 0.0):.4f}")

print("hold w1 = 0, nudge b:")
for b in [0.0, 0.5, 1.0]:
    print(f"   L(0, {b}) = {L(0.0, b):.4f}")

# Two different slopes at the same point, because they ask about different dials.
assert np.isclose(L(0.0, 0.0), 45.0)

## Step 4 — The Mathematics Under Test

$$\frac{\partial L}{\partial w_1} = 16.5(w_1-2) + 5.5(b-1)
\qquad
\frac{\partial L}{\partial b} = 5.5(w_1-2) + 2(b-1)$$

$$\nabla L = \left[\frac{\partial L}{\partial w_1},\; \frac{\partial L}{\partial b}\right]
\qquad
D_\mathbf{u}L = \nabla L \cdot \mathbf{u} = \|\nabla L\|\cos\theta$$

with the closed form the lecture derived, writing $u = w_1-2$ and $v = b-1$:

$$L = 8.25u^2 + 5.5uv + v^2$$

In [ ]:
# Step 5 — Manual calculation: the closed form and the partials (blog section 28).
def L_closed(w1, b):
    u, v = w1 - 2, b - 1
    return 8.25 * u**2 + 5.5 * u * v + v**2

print(f"{'(w1, b)':>12} {'direct':>12} {'closed form':>14}")
for w1, b in [(0., 0.), (1., .5), (2., 1.), (3., 2.), (1.5, .8)]:
    print(f"{str((w1, b)):>12} {L(w1, b):>12.6f} {L_closed(w1, b):>14.6f}")
    assert np.isclose(L(w1, b), L_closed(w1, b))
print("\nthe closed form is exact, not an approximation")

# Why 8.25 and 5.5? They are properties of the data alone.
assert np.isclose(np.mean(rooms ** 2), 8.25)      # coefficient of u^2
assert np.isclose(2 * np.mean(rooms), 5.5)        # coefficient of uv
print(f"mean(rooms^2) = {np.mean(rooms**2)},  2*mean(rooms) = {2*np.mean(rooms)}")

In [ ]:
# Step 5b — partial derivatives: analytic vs nudging one dial at a time.
def dL_dw1(w1, b):  return 16.5 * (w1 - 2) + 5.5 * (b - 1)
def dL_db(w1, b):   return 5.5 * (w1 - 2) + 2 * (b - 1)

def numeric_partial(f, w1, b, which, h=1e-6):
    if which == 0:
        return (f(w1 + h, b) - f(w1, b)) / h
    return (f(w1, b + h) - f(w1, b)) / h

print(f"{'(w1, b)':>12} {'dL/dw1':>10} {'numeric':>12} {'dL/db':>10} {'numeric':>12}")
for w1, b in [(0., 0.), (1., .5), (2., 1.), (3., 2.)]:
    a1, n1 = dL_dw1(w1, b), numeric_partial(L, w1, b, 0)
    a2, n2 = dL_db(w1, b), numeric_partial(L, w1, b, 1)
    print(f"{str((w1, b)):>12} {a1:>10.2f} {n1:>12.4f} {a2:>10.2f} {n2:>12.4f}")
    assert np.isclose(a1, n1, atol=1e-3) and np.isclose(a2, n2, atol=1e-3)

# At the origin the two partials differ by a factor of three.
assert dL_dw1(0., 0.) == -38.5 and dL_db(0., 0.) == -13.0
print("\nat (0,0): dL/dw1 = -38.5, dL/db = -13.0  -> w1 is the more sensitive dial")

# At the bottom of the bowl BOTH partials vanish.
assert np.isclose(dL_dw1(2., 1.), 0.0) and np.isclose(dL_db(2., 1.), 0.0)

In [ ]:
# Step 5c — the gradient really is the steepest direction (blog: directional derivative).
def grad(w1, b):
    return np.array([dL_dw1(w1, b), dL_db(w1, b)])

point = (0.0, 0.0)
g = grad(*point)
print("gradient at (0,0):", g, " norm:", round(float(np.linalg.norm(g)), 4))

# Try many unit directions; the directional derivative is grad . u
best_dir, best_rise = None, -np.inf
for t in np.linspace(0, 2 * np.pi, 720, endpoint=False):
    u = np.array([np.cos(t), np.sin(t)])
    rise = g @ u                       # D_u L
    if rise > best_rise:
        best_rise, best_dir = rise, u

print("steepest ASCENT direction found by search:", np.round(best_dir, 4))
print("gradient direction (normalized):          ", np.round(g / np.linalg.norm(g), 4))
assert np.allclose(best_dir, g / np.linalg.norm(g), atol=1e-2)
assert np.isclose(best_rise, np.linalg.norm(g), atol=1e-3)   # max rise = ||grad||
print("\nThe search agrees with the formula: steepest ascent IS the gradient,")
print("and the fastest possible rise equals ||grad|| exactly.")

In [ ]:
# Step 6 — First implementation: the general gradient formula, from scratch.
# dL/dw_j = (2/n) sum_i (y_hat_i - y_i) x_ij   and   dL/db = (2/n) sum_i (y_hat_i - y_i)
X = np.column_stack([rooms, area])

def gradient_general(w, b):
    err = (X @ w + b) - prices
    return (2 / len(X)) * (X.T @ err), 2 * err.mean()

# With w2 pinned at its true value, this must reproduce the 2-D partials above.
for w1, b in [(0., 0.), (1., .5), (3., 2.)]:
    gw, gb = gradient_general(np.array([w1, 0.005]), b)
    assert np.isclose(gw[0], dL_dw1(w1, b), atol=1e-9)
    assert np.isclose(gb, dL_db(w1, b), atol=1e-9)
print("the general formula reproduces the hand-derived partials exactly")

# And the explicit double loop matches the vectorized form (code mirrors math).
def gradient_explicit(w, b):
    n, d = X.shape
    g = np.zeros(d)
    for j in range(d):
        for i in range(n):
            g[j] += 2 * ((X[i] @ w + b) - prices[i]) * X[i, j]
    return g / n

w_test = np.array([1.0, 0.004])
assert np.allclose(gradient_explicit(w_test, 0.5), gradient_general(w_test, 0.5)[0])
print("explicit loop == vectorized gradient")

In [ ]:
# Step 7 — Visualization: the bowl, its contours, and the gradient field.
import matplotlib.pyplot as plt

w1s = np.linspace(-1, 5, 120)
bs  = np.linspace(-3, 5, 120)
W1, B = np.meshgrid(w1s, bs)
Z = 8.25 * (W1 - 2)**2 + 5.5 * (W1 - 2) * (B - 1) + (B - 1)**2

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

cs = axes[0].contour(W1, B, np.log10(Z + 1e-9), levels=22)
axes[0].plot(2, 1, "*", ms=18, color="crimson")
axes[0].set_title("log-loss contours: a tilted valley")
axes[0].set_xlabel("$w_1$"); axes[0].set_ylabel("$b$")

# gradient arrows point UPHILL, perpendicular to the contours
gw1, gb = np.meshgrid(np.linspace(-1, 5, 12), np.linspace(-3, 5, 12))
U = 16.5 * (gw1 - 2) + 5.5 * (gb - 1)
V = 5.5 * (gw1 - 2) + 2 * (gb - 1)
axes[1].contour(W1, B, np.log10(Z + 1e-9), levels=18, alpha=.4)
axes[1].quiver(gw1, gb, -U, -V, color="steelblue")     # minus = downhill
axes[1].plot(2, 1, "*", ms=18, color="crimson")
axes[1].set_title("negative gradient: every arrow points downhill")
axes[1].set_xlabel("$w_1$"); axes[1].set_ylabel("$b$")

# one slice each way through the point (0,0): different slopes, same point
axes[2].plot(w1s, [L(x, 0.0) for x in w1s], label="vary $w_1$, hold $b=0$")
axes[2].plot(bs, [L(0.0, x) for x in bs], label="vary $b$, hold $w_1=0$")
axes[2].set_title("two slices through the same point")
axes[2].set_xlabel("the dial being moved"); axes[2].legend(fontsize=8)

for ax in axes: ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
# Step 8 — The experiment: curvature is now a MATRIX.
# L = 8.25u^2 + 5.5uv + v^2, so the second derivatives are constants:
H = np.array([[16.5, 5.5],
              [5.5,  2.0]])
print("Hessian (the curvature matrix):\n", H)

evals = np.linalg.eigvalsh(H)
print(f"\neigenvalues: {np.round(evals, 4)}")
print(f"condition number: {evals[-1] / evals[0]:.1f}")
print(f"predicted stability limit: eta < 2/{evals[-1]:.4f} = {2/evals[-1]:.6f}")

assert np.isclose(evals[-1], 18.3501, atol=1e-3)   # steepest direction
assert np.isclose(evals[0], 0.1499, atol=1e-3)     # flattest direction
assert np.isclose(2 / evals[-1], 0.108991, atol=1e-5)

# Chapter 6 said eta < 2/L''. With many parameters, L'' becomes a matrix and the
# ceiling is set by its LARGEST eigenvalue -- the steepest direction of the bowl.

In [ ]:
# Step 9 — Change exactly one variable: the learning rate, against that prediction.
def descend(eta, steps=400):
    w1, b = 0.0, 0.0
    for _ in range(steps):
        w1, b = w1 - eta * dL_dw1(w1, b), b - eta * dL_db(w1, b)
        if not np.isfinite(w1) or abs(w1) > 1e10:
            return np.nan, np.nan
    return w1, b

print(f"predicted ceiling: {2/np.linalg.eigvalsh(H)[-1]:.6f}\n")
print(f"{'eta':>8} {'w1':>12} {'b':>12}   verdict")
for eta in [0.05, 0.10, 0.108, 0.109, 0.12]:
    with np.errstate(over="ignore", invalid="ignore"):
        w1, b = descend(eta)
    if not np.isfinite(w1):
        print(f"{eta:>8} {'overflow':>12} {'':>12}   diverged")
    else:
        ok = "converged" if abs(w1 - 2) < 1e-2 else "unstable"
        print(f"{eta:>8} {w1:>12.5f} {b:>12.5f}   {ok}")

w1_ok, _ = descend(0.108)
w1_bad, _ = descend(0.109)
assert abs(w1_ok - 2) < 1e-2            # just below the ceiling: fine
assert abs(w1_bad - 2) > 1e-2           # just above it: not fine

## Step 10 — Observe

Against your Step 2 predictions:

1. At $(0,0)$ the partials are $-38.5$ and $-13.0$. The loss is about **three times more
   sensitive to $w_1$ than to $b$** — the same point, two different slopes.
2. The steepest direction found by brute-force search over 720 directions agrees with
   $\nabla L/\|\nabla L\|$, and the fastest possible rise equals $\|\nabla L\|$ exactly.
3. The ceiling is set by the **largest** eigenvalue — the steepest direction. One learning
   rate has to survive the worst case.
4. Predicted $2/18.3501 = 0.108991$. The runs bracket it: $\eta = 0.108$ converges,
   $\eta = 0.109$ does not.

## Step 11 — Explain

**Why one number cannot describe the slope.** A bowl has a different slope in every
direction. A partial derivative answers "what if I move *only* this dial?", and the gradient
collects all those answers into one vector. It is not a slope — it is a *direction plus a
steepness*.

**Why the gradient is steepest.** The directional derivative is
$D_\mathbf{u}L = \nabla L\cdot\mathbf{u} = \|\nabla L\|\cos\theta$. Since $\cos\theta$ is
largest when $\theta = 0$, the fastest rise happens when you walk *along* the gradient, and
the fastest fall when you walk directly against it. That is the entire justification for
$\theta \leftarrow \theta - \eta\nabla L$.

**Why the ceiling comes from the largest eigenvalue.** Chapter 6 showed a single curvature
gives $\eta < 2/L''$. Here the curvature is the matrix $H$, and Chapter 5 told us a symmetric
matrix is just stretching along its eigenvector directions — by $18.35$ along one and
$0.1499$ along the other. The step must survive the steepest of those, so the ceiling is
$2/\lambda_{\max}$.

> And that same spread is the problem. The condition number here is $122$: the step small
> enough to be safe along the steep direction is far too small to make progress along the
> flat one. Chapter 2 met this as feature scaling; Chapter 32 fixes it with better optimizers.

In [ ]:
# Step 12 — Challenges.

# LEVEL 3 (Derive, then verify): show that the gradient is perpendicular to the contour
# line through a point. Pick a point, take a small step along the contour, and confirm
# the dot product with the gradient is near zero.

# YOUR CODE HERE


# LEVEL 4 (Investigate): unpin w2 so all THREE parameters are free.
# Build the 3x3 Hessian, find its largest eigenvalue, and predict the new ceiling.
# Chapter 2 showed the raw area feature makes this catastrophic -- how bad does it get?

# YOUR CODE HERE


# LEVEL 5 (Design): gradient descent takes the same size step in every direction.
# Design an update that takes BIG steps along flat directions and SMALL steps along
# steep ones. What would you need to know, and what would it cost to compute?
# (You are inventing the idea behind Newton's method and, loosely, Adam.)

## Step 13 — Reflection

- [ ] I can explain why "how steep is it here?" is incomplete without a direction.
- [ ] I computed a partial derivative by nudging one dial and holding the other.
- [ ] I confirmed by brute-force search that the gradient really is the steepest direction.
- [ ] I can state the chain rule and say why Chapter 27 will need it.
- [ ] I predicted a learning-rate ceiling from an eigenvalue before running anything.
- [ ] I can explain what a condition number of 122 means for training.

### The question this chapter leaves open

We know which way is downhill, and how steep it is. We still do not know **how far to step,
or how many times**, or whether repeating the step arrives anywhere at all. The gradient is
honest only about the next instant.

➡️ **Next:** [Chapter 08 — Gradient Descent: Teaching a Model to Improve](<../Lecture 08 - Gradient Descent: Teaching a Model to Improve/blog.md>)